In [1]:
import pandas as pd
import sys
sys.path.append('../../')
from MEGA_utilities import data_col_standardize, data_remove_duplicate

# load data
origin_df = pd.read_csv('../../../data/processed/0.1_Final_df.csv')

## DNA MEGA

In [9]:
## select required columns
DNA_data_filt1 = origin_df[['serumName','virusName','serumHA_DNA','virusHA_DNA','serumPassCat','virusPassCat',
                            'serumType','HI_Dist']].copy()
## remove duplicated row and mean HI_Dist
DNA_data_filt2 = DNA_data_filt1.groupby(['serumHA_DNA', 'virusHA_DNA', 'serumPassCat', 'virusPassCat']) \
        .agg({'serumName': 'first', 'virusName': 'first', 'serumType': 'first', 'HI_Dist': 'mean'}) \
        .reset_index()[['serumName', 'virusName', 'serumHA_DNA', 'virusHA_DNA',
                        'serumPassCat', 'virusPassCat', 'serumType', 'HI_Dist']]
## remove PassCat = 'BOTH'
DNA_data_filt3 = DNA_data_filt2[(DNA_data_filt2['serumPassCat'] != 'BOTH') &
                                (DNA_data_filt2['virusPassCat'] != 'BOTH')].reset_index(drop=True)
## replace PassCat to special token
DNA_data_filt4 = DNA_data_filt3.replace({'serumPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'},
                                         'virusPassCat': {'EGG': '<EGG>', 'CELL': '<CELL>'}})

In [4]:
import torch
from torch.utils.data import Dataset, DataLoader

class DNADataset(Dataset):
    def __init__(self, DataFrame):
        self.sequence = (DataFrame['serumHA_DNA'] + '<eos>' + DataFrame['virusHA_DNA'] + \
            '<eos>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat']).tolist()
        self.labels = torch.tensor(DataFrame['HI_Dist'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.sequence[idx], self.labels[idx]

In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(DNA_data_filt4, test_size=0.1, random_state=42)
train_df, valid_df = train_test_split(train_df, test_size=1/9, random_state=42)

train_dataset = DNADataset(train_df)
valid_dataset = DNADataset(valid_df)
test_dataset = DNADataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [6]:
# train_df.to_csv('../../../data/processed/1.1/DNA_train_df.csv', index=False)
# valid_df.to_csv('../../../data/processed/1.1/DNA_valid_df.csv', index=False)
# test_df.to_csv('../../../data/processed/1.1/DNA_test_df.csv', index=False)

In [5]:
from bio_tokenizer import BioTokenizer
from transformers import MegaConfig, MegaForSequenceClassification
from MEGA_utilities import count_parameters
from torch.optim import AdamW
from transformers import get_scheduler
import torch

# get tokenizer and model
tokenizer = BioTokenizer(vocab_file='./vocab_DNA.txt')

# update the num_vocab and num_label
config = MegaConfig()
config.num_labels=1
config.vocab_size=22
config.max_positions=4000
config.num_attention_heads=4
config.num_hidden_layers=4
device = torch.device("cuda:0")
model = MegaForSequenceClassification(config)
model.to(device)
print("Number of parameters: %e"%count_parameters(model))

# optimizer
optimizer = AdamW(model.parameters(), lr=5e-4)

# scheduler
num_epochs = 160
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(name="linear", optimizer=optimizer,
                             num_warmup_steps=len(train_loader), num_training_steps=num_training_steps)

Number of parameters: 9.116250e+05


In [6]:
from tqdm import tqdm
from utilities import print_exams
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
from utilities import EarlyStopping
from datetime import datetime

progress_bar = tqdm(range(num_training_steps))
early_stopping = EarlyStopping(patience=10, delta=0.005, save_dir='./1.1_DNA_model/')

# Training loop
for epoch in range(num_epochs):
    model.train()
    loss_ls = []
    for batch_seq, batch_label in train_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)

        loss = outputs.loss
        loss_ls.append(loss.item())

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)
    train_loss = sum(loss_ls) / len(loss_ls)
    print('train loss :', train_loss)

    prediction_ls = []
    reference_ls = []
    logits_ls = []
    loss_ls_valid = []
    with torch.no_grad():
        model.eval()
        for batch_seq, batch_label in valid_loader:
            batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
            batch_input = batch_input.to(device)
            batch_label = batch_label.to(device)

            outputs = model(**batch_input, labels=batch_label)
            logits = outputs.logits
            loss = outputs.loss

            logits_ls.append(logits)
            loss_ls_valid.append(loss.item())
            prediction_ls += logits.tolist()
            prediction_ls_final = []
            for sublist in prediction_ls:
                for element in sublist:
                    prediction_ls_final.append(element)
            reference_ls += batch_label.tolist()

    print_exams(prediction_ls_final, reference_ls)
    valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
    valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
    valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
    valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic
    
    early_stopping(valid_mse, model)
    if early_stopping.early_stop:
        print("Early stopping")
        break

    ## 将epoch信息写入log.txt
    with open('./1.1_DNA_model/log.txt', 'a') as f:
        current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"[{current_time}] Epoch {epoch + 1}/{num_epochs}, train loss: {train_loss:.4f}, valid MAE: {valid_MAE:.4f}, valid MSE: {valid_mse:.4f}, valid Pearson: {valid_pearson:.4f}, valid Spearman: {valid_spearman:.4f}\n")

  1%|          | 6552/1048320 [18:04<48:39:39,  5.95it/s]

train loss : 3.2367296095992075
MAE:  1.3764986898320226
MSE:  3.035418865597516
pearson correlation:  PearsonRResult(statistic=0.43956092772381766, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.4070784760476371, pvalue=0.0)
Validation MSE decrease (inf --> 3.035419).  Saving model ...


  1%|▏         | 13104/1048320 [38:15<47:54:34,  6.00it/s]  

train loss : 2.9657495072809694
MAE:  1.2871324684550491
MSE:  2.778214997846419
pearson correlation:  PearsonRResult(statistic=0.5114418749341693, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.49452698708945003, pvalue=0.0)
Validation MSE decrease (3.035419 --> 2.778215).  Saving model ...


  2%|▏         | 19656/1048320 [58:26<48:23:18,  5.91it/s]   

...

train loss : 0.6934000751895145


 83%|████████▎ | 871417/1048320 [44:28:08<1750:55:14, 35.63s/it]

MAE:  0.6966360792751914
MSE:  0.8284857953415604
pearson correlation:  PearsonRResult(statistic=0.8832646366332357, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8453885258275632, pvalue=0.0)
EarlyStopping counter: 9 out of 10


 84%|████████▍ | 877968/1048320 [44:45:54<7:18:07,  6.48it/s]   

train loss : 0.690394003201226
MAE:  0.6962539811935889
MSE:  0.8238660025107356
pearson correlation:  PearsonRResult(statistic=0.8839588758890282, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8454090665694308, pvalue=0.0)
EarlyStopping counter: 10 out of 10
Early stopping


In [7]:
prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
with torch.no_grad():
    model.eval()
    for batch_seq, batch_label in test_loader:
        batch_input = tokenizer(batch_seq, padding='longest', return_tensors="pt")
        batch_input = batch_input.to(device)
        batch_label = batch_label.to(device)

        outputs = model(**batch_input, labels=batch_label)
        logits = outputs.logits
        loss = outputs.loss

        logits_ls.append(logits)
        loss_ls_valid.append(loss.item())
        prediction_ls += logits.tolist()
        prediction_ls_final = []
        for sublist in prediction_ls:
            for element in sublist:
                prediction_ls_final.append(element)
        reference_ls += batch_label.tolist()

valid_MAE = mean_absolute_error(reference_ls, prediction_ls_final)
valid_mse = mean_squared_error(reference_ls, prediction_ls_final)
valid_pearson = pearsonr(reference_ls, prediction_ls_final).statistic
valid_spearman = spearmanr(reference_ls, prediction_ls_final).statistic

In [8]:
print(valid_MAE)
print(valid_mse)
print(valid_pearson)
print(valid_spearman)

0.6905446622852732
0.8177329647194344
0.8826784831007666
0.8467081312444236
